In [1]:
import os
import sys

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

In [2]:
flu_obj_cy5_1 =  fl.Fluorophore(name="testfluo_1", position=[0, 0])

flu_obj_atto643 = fl.Fluorophore(name="testfluo_2", position=[0, 2])

In [3]:
flu_sys_1xcy5_1xatto643 = fl.FluorophoreSystem(fluorophores=[flu_obj_cy5_1, flu_obj_atto643])

In [4]:
transitions = flu_sys_1xcy5_1xatto643.load_transitions(
    irradiance=2, wavelength=640, bleaching=True, energy_transfer=True, dstorm=False
)
tset = tr.TransitionSet(
    transitions=transitions, fluorophore_system=flu_sys_1xcy5_1xatto643
)
tset.finalize()
tr_set_bl_et_2f_diff = tset

In [6]:
tr_set_bl_et_2f_diff.transition_df

transition_type  \
Fluorophore                             identity                                           
testfluo_1                              0                      TransitionType.EXCITATION   
                                        1            TransitionType.FLUORESCENT_EMISSION   
                                        2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                        3         TransitionType.INTERSYSTEM_CROSSING_TS   
                                        4                   TransitionType.ISOMERIZATION   
                                        5               TransitionType.BACKISOMERIZATION   
                                        6           TransitionType.INTERNAL_CONVERSION_S   
                                        7                TransitionType.PHOTOBLEACHING_1   
D: testfluo_1, A: testfluo_2, dist: 2.0 8                            TransitionType.FRET   
testfluo_2                              9                      TransitionType.EXCITATION   
                                        10           TransitionType.FLUORESCENT_EMISSION   
                                        11        TransitionType.INTERSYSTEM_CROSSING_ST   
                                        12        TransitionType.INTERSYSTEM_CROSSING_TS   
                                        13          TransitionType.INTERNAL_CONVERSION_S   
                                        14               TransitionType.PHOTOBLEACHING_1   
D: testfluo_2, A: testfluo_1, dist: 2.0 15                           TransitionType.FRET   

                                                 abbreviation  \
Fluorophore                             identity                
testfluo_1                              0                 EXC   
                                        1                 FLU   
                                        2               ISCST   
                                        3               ISCTS   
                                        4                 ISO   
                                        5                BISO   
                                        6                 ICS   
                                        7                BLE1   
D: testfluo_1, A: testfluo_2, dist: 2.0 8                FRET   
testfluo_2                              9                 EXC   
                                        10                FLU   
                                        11              ISCST   
                                        12              ISCTS   
                                        13                ICS   
                                        14               BLE1   
D: testfluo_2, A: testfluo_1, dist: 2.0 15               FRET   

                                                      initial_state  \
Fluorophore                             identity                      
testfluo_1                              0            SingleState.S0   
                                        1            SingleState.S1   
                                        2            SingleState.S1   
                                        3            SingleState.T1   
                                        4            SingleState.S1   
                                        5           SingleState.Cis   
                                        6            SingleState.S1   
                                        7            SingleState.T1   
D: testfluo_1, A: testfluo_2, dist: 2.0 8         PairedState.S1_S0   
testfluo_2                              9            SingleState.S0   
                                        10           SingleState.S1   
                                        11           SingleState.S1   
                                        12           SingleState.T1   
                                        13           SingleState.S1   
                                        14           SingleState.T1   
D: testfluo_2, A: testfluo_1, dist: 2.

In [5]:
pr.Prediction(tr_set_bl_et_2f_diff)

WARNING for line: pr.Prediction(tr_set_bl_et_2f_diff)
 prediction accuracy of energy transfers more difficult to tune. Only frequencies available, lifetimes and occupations not available. 
WARNING for line: pr.Prediction(tr_set_bl_et_2f_diff)
 absorbing states have a lifetime of inf and a frequency / occupation of 0. Absorbing transitions have a frequency of 0. 


In [3]:
fluorophores = fl.construct_fluorophores(
    name="cy5_widengren", distance=10, count=2, shape="square"
)

fluorophore_system = fl.FluorophoreSystem(fluorophores)

In [5]:
transi = fluorophore_system.load_transitions()

In [6]:
transition_set = tr.TransitionSet(transi, fluorophore_system)

In [7]:
transition_set.finalize()

In [8]:
simu = si.Simulation(transition_set)

In [9]:
simu.run()

In [19]:
emis = em.Emissions(bandpass=[200, 400])

In [21]:
emis.extract(simu)